In [23]:
! pip install -qU langchain-text-splitters
! pip install pandas
! pip install -qU langchain-huggingface
! pip install sentence-transformers
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 12.7 MB/s  0:00:00 eta 0:00:01


In [24]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
df=pd.read_csv("../data/processed/cleaned_data.csv")
print(df.shape)
print(df['context_text'].head())
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)
texts=text_splitter.create_documents(df["context_text"].astype(str).tolist())
print(texts[0])
print(texts[1])


(211269, 8)
0    Chronic rhinosinusitis (CRS) is a heterogeneou...
1    Phosphatidylethanolamine N-methyltransferase (...
2    Psammaplin A (PsA) is a natural product isolat...
3    This study examined links between DNA methylat...
4    Tumor microenvironment immunity is associated ...
Name: context_text, dtype: object
page_content='Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoin

In [25]:

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings":True},
)
query_result=embeddings.embed_query("what is treatment for diabetes")
print("Query size:",len(query_result))
doc_result=embeddings.embed_documents(df["context_text"].head(5).astype(str).tolist()
)
print("Docs embedded:",len(doc_result))
print("Each doc size",len(doc_result[0]))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8168.11it/s]


Query size: 768
Docs embedded: 5
Each doc size 768


In [32]:
import faiss
import numpy as np
vectors=np.load('../data/processed/embeddings.npy').astype('float32')
print(vectors.shape)
dim=vectors.shape[1]
index=faiss.IndexFlatL2(dim)
index.add(vectors)
print("Total vectors",index.ntotal)
faiss.write_index(index,'../data/processed/faiss_index')


(211269, 384)
Total vectors 211269
